In [2]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [3]:
train_data= pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

### Data- Preprocessing

In [4]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text)
    text = re.sub(r"\s+"," ",text)
    text = re.sub(r"<.*?>"," ",text)
    text.strip().lower()
    return text 

In [5]:
train_data["dialogue"] = train_data["dialogue"].fillna("").astype(str).apply(clean_data)
train_data["summary"] = train_data["summary"].fillna("").astype(str).apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].fillna("").astype(str).apply(clean_data)
val_data["summary"] = val_data["summary"].fillna("").astype(str).apply(clean_data)

In [6]:
train_data["dialogue"][0]

"Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)"

### Tokenize

In [7]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [8]:
# raw data =>tokenize 

def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets = tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [9]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

### Working with the Model

In [10]:
#NLP =>Generation task


model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [11]:
import torch 

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(device)

cuda


In [12]:
#Training Arguments 

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=10,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps = 500,
)

In [13]:
trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [14]:
#Train the module
trainer.train()

  0%|          | 0/18420 [00:00<?, ?it/s]

{'loss': 4.0641, 'grad_norm': 0.546657145023346, 'learning_rate': 5e-05, 'epoch': 0.27}
{'loss': 0.423, 'grad_norm': 0.6590235233306885, 'learning_rate': 4.860491071428572e-05, 'epoch': 0.54}
{'loss': 0.4026, 'grad_norm': 0.595012903213501, 'learning_rate': 4.720982142857143e-05, 'epoch': 0.81}


  0%|          | 0/103 [00:00<?, ?it/s]

{'eval_loss': 0.34803035855293274, 'eval_runtime': 10.2087, 'eval_samples_per_second': 80.128, 'eval_steps_per_second': 10.089, 'epoch': 1.0}
{'loss': 0.3851, 'grad_norm': 0.4429655373096466, 'learning_rate': 4.581473214285715e-05, 'epoch': 1.09}
{'loss': 0.3784, 'grad_norm': 0.5584485530853271, 'learning_rate': 4.4419642857142854e-05, 'epoch': 1.36}
{'loss': 0.3795, 'grad_norm': 0.6417127251625061, 'learning_rate': 4.302455357142857e-05, 'epoch': 1.63}
{'loss': 0.367, 'grad_norm': 0.7056760191917419, 'learning_rate': 4.162946428571429e-05, 'epoch': 1.9}


  0%|          | 0/103 [00:00<?, ?it/s]

...

  0%|          | 0/103 [00:00<?, ?it/s]

{'eval_loss': 0.32028815150260925, 'eval_runtime': 11.8296, 'eval_samples_per_second': 69.149, 'eval_steps_per_second': 8.707, 'epoch': 9.0}
{'loss': 0.3221, 'grad_norm': 0.49981728196144104, 'learning_rate': 3.9620535714285715e-06, 'epoch': 9.23}
{'loss': 0.3211, 'grad_norm': 0.39431068301200867, 'learning_rate': 2.5669642857142856e-06, 'epoch': 9.5}
{'loss': 0.3283, 'grad_norm': 0.5396395921707153, 'learning_rate': 1.1718750000000001e-06, 'epoch': 9.77}


  0%|          | 0/103 [00:00<?, ?it/s]

{'eval_loss': 0.3196778893470764, 'eval_runtime': 12.6271, 'eval_samples_per_second': 64.781, 'eval_steps_per_second': 8.157, 'epoch': 10.0}
{'train_runtime': 7325.6059, 'train_samples_per_second': 20.11, 'train_steps_per_second': 2.514, 'train_loss': 0.44619020592506237, 'epoch': 10.0}


TrainOutput(global_step=18420, training_loss=0.44619020592506237, metrics={'train_runtime': 7325.6059, 'train_samples_per_second': 20.11, 'train_steps_per_second': 2.514, 'total_flos': 1.993855419285504e+16, 'train_loss': 0.44619020592506237, 'epoch': 10.0})